In [ ]:
load_best_model_at_end=True, metric_for_best_model='f1_macro', greater_is_better=True, and use save_strategy='epoch'

In [ ]:
# ============================================================
# Load best saved prompt model for inference
# ============================================================
import json

best_checkpoint = None
if 'trainer' in globals() and getattr(trainer, 'state', None) is not None:
    best_checkpoint = getattr(trainer.state, 'best_model_checkpoint', None)

if best_checkpoint and os.path.isdir(best_checkpoint):
    BEST_PRED_MODEL_DIR = best_checkpoint
else:
    BEST_PRED_MODEL_DIR = BEST_MODEL_DIR

if not os.path.isdir(BEST_PRED_MODEL_DIR):
    raise FileNotFoundError(f'Best model directory not found: {BEST_PRED_MODEL_DIR}')

vc_path = os.path.join(BEST_PRED_MODEL_DIR, 'verbalizer_config.json')
if not os.path.exists(vc_path):
    raise FileNotFoundError(f'Missing verbalizer config: {vc_path}')

with open(vc_path, 'r') as f:
    vc = json.load(f)

loaded_verbalizer_ids = {int(k): v for k, v in vc['verbalizer_ids'].items()}
loaded_mask_token_id = int(vc.get('mask_token_id', MASK_TOKEN_ID))
loaded_prompt_template = vc.get('prompt_template', PROMPT_TEMPLATE)

pred_tokenizer = AutoTokenizer.from_pretrained(BEST_PRED_MODEL_DIR)
pred_model = PromptMLMClassifier(
    model_name=BEST_PRED_MODEL_DIR,
    verbalizer_ids=loaded_verbalizer_ids,
    mask_token_id=loaded_mask_token_id,
).to(device)
pred_model.eval()

print(f'Loaded model from: {BEST_PRED_MODEL_DIR}')
print(f'Mask token for inference: {pred_tokenizer.mask_token!r} (id={pred_tokenizer.mask_token_id})')


In [ ]:
# ============================================================
# Build helpers + generate dev.txt using best tuned model
# ============================================================
DEV_TXT_PATH = 'dev.txt'
TEST_TXT_PATH = 'test.txt'
PRED_BATCH_SIZE = 32

def prepare_prompted_frame(frame, prompt_template, mask_token):
    frame = frame.copy()
    frame['keyword'] = frame['keyword'].fillna('').astype(str)
    frame['country'] = frame['country'].fillna('').astype(str)
    frame['text'] = frame['text'].fillna('').astype(str)
    frame['clean_text'] = frame['text'].apply(preprocess_text)
    frame['model_text'] = (
        frame['keyword'].str.strip()
        + ' </s> ' + frame['country'].str.strip()
        + ' </s> ' + frame['clean_text'].str.strip()
    )
    frame['prompted_text'] = frame['model_text'].apply(
        lambda t: prompt_template.format(text=t, mask=mask_token)
    )
    return frame

def predict_binary_labels(prompted_texts, model, tokenizer, batch_size=32, max_length=256):
    preds = []
    total = len(prompted_texts)
    for start in range(0, total, batch_size):
        batch_texts = prompted_texts[start:start + batch_size]
        batch = tokenizer(
            batch_texts,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors='pt',
        )
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.no_grad():
            logits = model(**batch).logits

        batch_preds = torch.argmax(logits, dim=-1).cpu().tolist()
        preds.extend(int(p) for p in batch_preds)

    return preds

# Dev set in official dev-id file order
dev_input = dev_ids_df[['par_id']].copy()
dev_input['par_id'] = dev_input['par_id'].astype(str)
dev_lookup_cols = ['par_id', 'keyword', 'country', 'text']
dev_lookup = df[dev_lookup_cols].drop_duplicates(subset=['par_id'], keep='first')
dev_input = dev_input.merge(dev_lookup, on='par_id', how='left')

if dev_input['text'].isna().any():
    missing = int(dev_input['text'].isna().sum())
    raise ValueError(f'Missing text for {missing} dev par_id rows after merge.')

dev_prompted = prepare_prompted_frame(
    dev_input,
    prompt_template=loaded_prompt_template,
    mask_token=pred_tokenizer.mask_token,
)

dev_preds = predict_binary_labels(
    dev_prompted['prompted_text'].tolist(),
    model=pred_model,
    tokenizer=pred_tokenizer,
    batch_size=PRED_BATCH_SIZE,
    max_length=MAX_LENGTH,
)

if not set(dev_preds).issubset({0, 1}):
    raise ValueError('Dev predictions contain values outside {0,1}.')
if len(dev_preds) != len(dev_input):
    raise ValueError(f'Dev prediction count mismatch: {len(dev_preds)} vs {len(dev_input)}')

with open(DEV_TXT_PATH, 'w') as f:
    f.write('\n'.join(str(p) for p in dev_preds))
    f.write('\n')

print(f'Dev rows: {len(dev_input)} | Predictions written: {len(dev_preds)} -> {DEV_TXT_PATH}')


In [ ]:
# ============================================================
# Train on full labeled data (train+dev) and generate test.txt
# ============================================================
FULL_TRAIN_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'full_train_final')
FULL_TRAIN_POS_REPLICAS = 8
os.makedirs(FULL_TRAIN_OUTPUT_DIR, exist_ok=True)

full_train_base_df = df.copy().reset_index(drop=True)
replicated_full_pcls = pd.concat(
    [full_train_base_df[full_train_base_df['label'] == 1]] * FULL_TRAIN_POS_REPLICAS,
    ignore_index=True,
)
full_train_df = pd.concat([full_train_base_df, replicated_full_pcls], ignore_index=True)
full_train_df = full_train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Full train (base): {len(full_train_base_df):,} rows')
print(f'Full train (+OS):  {len(full_train_df):,} rows')

full_prompted = prepare_prompted_frame(
    full_train_df[['keyword', 'country', 'text', 'label']].copy(),
    prompt_template=loaded_prompt_template,
    mask_token=pred_tokenizer.mask_token,
)

full_train_hf = Dataset.from_pandas(
    full_prompted[['prompted_text', 'label']].rename(columns={'prompted_text': 'text'}),
    preserve_index=False,
)

def tokenize_for_full_train(batch):
    return pred_tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

full_train_ds = full_train_hf.map(tokenize_for_full_train, batched=True, remove_columns=['text'])
full_train_ds = full_train_ds.rename_column('label', 'labels')
full_train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

full_model = PromptMLMClassifier(
    model_name=BEST_PRED_MODEL_DIR,
    verbalizer_ids=loaded_verbalizer_ids,
    mask_token_id=loaded_mask_token_id,
).to(device)

full_training_args = TrainingArguments(
    output_dir=FULL_TRAIN_OUTPUT_DIR,
    num_train_epochs=2,
    save_total_limit=2,
    learning_rate=1e-5,
    eval_strategy='no',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    warmup_steps=2000,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    report_to='none',
    seed=SEED,
)

full_trainer = PromptTrainer(
    model=full_model,
    args=full_training_args,
    train_dataset=full_train_ds,
    eval_dataset=None,
    processing_class=pred_tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=pred_tokenizer),
    compute_metrics=None,
)

full_train_result = full_trainer.train()
print(full_train_result)
full_model.eval()


In [ ]:
FULL_FINAL_MODEL_DIR = os.path.join(FULL_TRAIN_OUTPUT_DIR, 'final')
os.makedirs(FULL_FINAL_MODEL_DIR, exist_ok=True)
full_model.mlm.save_pretrained(FULL_FINAL_MODEL_DIR)
pred_tokenizer.save_pretrained(FULL_FINAL_MODEL_DIR)
with open(os.path.join(FULL_FINAL_MODEL_DIR, 'verbalizer_config.json'), 'w') as f:
    json.dump({
        'verbalizer': {str(k): v for k, v in VERBALIZER.items()},
        'verbalizer_ids': {str(k): v for k, v in loaded_verbalizer_ids.items()},
        'prompt_template': loaded_prompt_template,
        'mask_token_id': loaded_mask_token_id,
    }, f, indent=2)
print(f'Saved full-trained model to: {FULL_FINAL_MODEL_DIR}')

# Test set in original file order
dpm.load_test()
test_input = dpm.test_set_df.copy()
test_prompted = prepare_prompted_frame(
    test_input,
    prompt_template=loaded_prompt_template,
    mask_token=pred_tokenizer.mask_token,
)

test_preds = predict_binary_labels(
    test_prompted['prompted_text'].tolist(),
    model=full_model,
    tokenizer=pred_tokenizer,
    batch_size=PRED_BATCH_SIZE,
    max_length=MAX_LENGTH,
)

if not set(test_preds).issubset({0, 1}):
    raise ValueError('Test predictions contain values outside {0,1}.')
if len(test_preds) != len(test_input):
    raise ValueError(f'Test prediction count mismatch: {len(test_preds)} vs {len(test_input)}')

with open(TEST_TXT_PATH, 'w') as f:
    f.write('\n'.join(str(p) for p in test_preds))
    f.write('\n')

print(f'Test rows: {len(test_input)} | Predictions written: {len(test_preds)} -> {TEST_TXT_PATH}')
print(f'Test expected (example target): 3832 | Actual loaded: {len(test_input)}')


In [ ]:
# Quick sanity checks on output files
for path in ['dev.txt', 'test.txt']:
    with open(path, 'r') as f:
        lines = [line.strip() for line in f if line.strip() != '']
    uniq = sorted(set(lines))
    print(f'{path}: lines={len(lines)}, unique_labels={uniq}')
